# Lekce 11 - Protokol kontextu modelu (MCP)

The **Protokol kontextu modelu (MCP)** je otevřený standard, který umožňuje agentům dynamicky objevovat a používat nástroje, prostředky a zdroje dat za běhu. Místo pevného zakódování nástrojů do agenta umožňuje MCP agentům připojovat se k externím serverům, které na požádání zpřístupňují své schopnosti.

V této lekci se naučíte:
- Co je MCP a proč je důležitý pro agentní systémy
- Jak funguje klient-server architektura MCP
- Jak vytvářet agenty, kteří používají objevování nástrojů ve stylu MCP


## Nastavení

**Požadavky:**
- Projekt Azure AI Foundry s nasazeným modelem
- Spusťte `az login` pro autentizaci


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Co je Model Context Protocol (MCP)?

MCP definuje standardní způsob, jak mohou AI agenti objevovat a interagovat s externími nástroji a zdroji dat:

- **MCP Server**: Zpřístupňuje nástroje, zdroje a výzvy prostřednictvím standardního protokolu
- **MCP Client**: Běhové prostředí agenta, které se připojuje k serverům a objevuje dostupné schopnosti
- **Dynamic Discovery**: Agentům nejsou potřeba pevně zakódované nástroje — při běhu objevují, co je k dispozici

To je velmi užitečné při vytváření rozšiřitelných systémů agentů, kde lze přidávat nové schopnosti bez úprav kódu agenta.


## Jak MCP funguje

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. Agent (MCP klient) se připojí k MCP serveru
2. Server odpoví seznamem dostupných nástrojů a jejich schémat
3. Agent pak může během svého uvažování zavolat jakýkoli nalezený nástroj
4. Výsledky se vracejí zpět stejným protokolem


## Simulace zjišťování nástrojů MCP

Protože skutečný MCP server vyžaduje běžící serverový proces, předvedeme tento vzor pomocí funkcí `@tool`, které simulují to, co by poskytla ubytovací služba připojená k MCP.

V produkčním prostředí by byly tyto nástroje dynamicky zjišťovány z MCP serveru, nikoli definovány lokálně.


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## Vytváření agenta s nástroji ve stylu MCP


In [ ]:
agent = await provider.create_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## MCP v produkci

V produkčním prostředí MCP umožňuje výkonné vzory:

- **Dynamic tool discovery**: Dynamické objevování nástrojů za běhu
- **Decoupled architecture**: Oddělená architektura — poskytovatelé nástrojů se mohou aktualizovat nezávisle na agentovi
- **Cross-organization sharing**: Sdílení napříč organizacemi — týmy mohou vystavovat schopnosti přes MCP servery, které může použít kterýkoli agent
- **Microsoft Agent Framework support**: Podpora Microsoft Agent Framework — MAF má vestavěnou podporu klienta MCP prostřednictvím integrace `mcp`

To use a real MCP server with MAF, you would connect via `hosted_mcp_tool()` or the MCP client integration.

**Learn more:**
- [MCP Specification](https://modelcontextprotocol.io/)
- [Podpora MCP v Microsoft Agent Framework](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## Shrnutí

V této lekci jste se naučili:
- **MCP** je otevřený standard pro dynamické objevování nástrojů mezi agenty a poskytovateli nástrojů
- **architektura klient-server** umožňuje agentům objevovat schopnosti během běhu programu
- MCP umožňuje **rozšiřitelné, oddělené agentní systémy**, kde lze nástroje přidávat bez změn kódu
- Microsoft Agent Framework poskytuje **vestavěnou podporu MCP** pro produkční použití


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Prohlášení o vyloučení odpovědnosti**:
Tento dokument byl přeložen pomocí AI překladatelské služby [Co-op Translator](https://github.com/Azure/co-op-translator). I když usilujeme o přesnost, mějte prosím na paměti, že automatické překlady mohou obsahovat chyby nebo nepřesnosti. Původní dokument v jeho originálním jazyce by měl být považován za závazný zdroj. Pro důležité informace doporučujeme využít profesionální lidský překlad. Nejsme odpovědní za jakákoli nedorozumění nebo chybné výklady vyplývající z použití tohoto překladu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
